# Train the filament U-Net (ImageNet-pretrained encoder) on a Kaggle GPU
Settings (right panel): **Accelerator → GPU T4 x2** and **Internet → On** (needed to download the pretrained encoder).
Inputs: the competition data and your `filseg-code` dataset.
Run all cells (**Run All**). Training is limited to `MINUTES`. When it finishes, download `unet_results.zip` from **Output** (right panel).

In [ ]:
import glob, os, shutil, subprocess, sys
# find the competition data and the uploaded code, wherever Kaggle mounted them
JSON = glob.glob("/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json", recursive=True)[0]
ROOT = os.path.dirname(os.path.dirname(JSON))                     # .../MAGFiLO_1.0_Kaggle_2026
TRAIN_DIR = os.path.join(ROOT, "train", "train_images")
TEST_DIR  = os.path.join(ROOT, "test", "test_images")
RUN = glob.glob("/kaggle/input/**/scripts/run.py", recursive=True)[0]
CODE = os.path.dirname(os.path.dirname(RUN))
shutil.copytree(CODE, "/kaggle/working/filseg", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("data", "*.zip", "__pycache__"))
os.chdir("/kaggle/working/filseg")
os.makedirs("weights", exist_ok=True)
print(JSON, TRAIN_DIR, TEST_DIR, os.getcwd(), sep="\n")
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
MINUTES = 150   # training time budget; Kaggle sessions can run up to 12 h
ARGS = f'--train-json "{JSON}" --train-dir "{TRAIN_DIR}"'
# ImageNet-pretrained ResNet-34 encoder, larger crops and batch on the GPU
MODEL = "--arch smp --encoder resnet34 --crop 512 --batch 16 --lr 5e-4"

In [ ]:
MINUTES = 150   # training time budget; Kaggle sessions can run up to 12 h
ARGS = f'--train-json "{JSON}" --train-dir "{TRAIN_DIR}"'

!python scripts/run.py train --data /kaggle/working {ARGS} {MODEL} --minutes {MINUTES}

In [ ]:
!python scripts/run.py train --data /kaggle/working {ARGS} --minutes {MINUTES}

## 2. Tune threshold / fragment linking for PQ on the validation split

In [ ]:
!python scripts/run.py tune --method unet {ARGS} --n-val 60

## 3. Submission file

In [ ]:
!python scripts/run.py submit --method unet --test-dir "{TEST_DIR}" --out /kaggle/working/submission.csv

## 4. Files to download (Output panel → /kaggle/working/filseg/weights) and copy into `Documents\\filament-seg\\weights`

In [ ]:
shutil.copy("/kaggle/working/submission.csv", "/kaggle/working/filseg/weights/submission_unet.csv")
for f in sorted(os.listdir("weights")): print(f, os.path.getsize("weights/" + f) // 1024, "KB")
shutil.make_archive("/kaggle/working/unet_results", "zip", "weights")   # one zip with everything
print("Download /kaggle/working/unet_results.zip")